下面这个 Jupyter 代码会扫描 raw 文件夹下所有检查文件夹，也就是文件夹名前 8 位是合法日期 YYYYMMDD的文件夹。
然后只删除这些检查文件夹直接下面的：
images    
with_data_images    
without_data_images

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import os
import stat


# =========================
# 1. 修改成你的 raw 根目录
# =========================
RAW_ROOT = Path(r"G:\wxy_codes\Fetal_hydronephrosis\1第一批数据\batch1data_4\raw")


# =========================
# 2. 是否只预览
# True：只打印将要删除的文件夹，不真正删除
# False：真正删除
# =========================
DRY_RUN = False


# =========================
# 3. 需要删除的文件夹名字
# =========================
TARGET_DIR_NAMES = {
    "images",
    "with_data_images",
    "without_data_images",
}


def is_exam_folder(folder: Path) -> bool:
    """
    判断是否为检查文件夹：
    文件夹名前 8 位是合法日期 YYYYMMDD。
    例如：202412300188 -> 前 8 位是 20241230
    """
    if not folder.is_dir():
        return False

    name = folder.name

    if len(name) < 8:
        return False

    date_part = name[:8]

    if not date_part.isdigit():
        return False

    try:
        datetime.strptime(date_part, "%Y%m%d")
        return True
    except ValueError:
        return False


def handle_remove_readonly(func, path, exc_info):
    """
    处理 Windows 下只读文件导致无法删除的问题。
    """
    os.chmod(path, stat.S_IWRITE)
    func(path)


def delete_target_dirs(raw_root: Path, dry_run: bool = True):
    if not raw_root.exists():
        raise FileNotFoundError(f"RAW_ROOT 不存在: {raw_root}")

    exam_folders = []
    target_dirs = []

    # 递归查找所有检查文件夹
    for folder in raw_root.rglob("*"):
        if is_exam_folder(folder):
            exam_folders.append(folder)

            # 只检查检查文件夹的直接子文件夹
            for target_name in TARGET_DIR_NAMES:
                target_dir = folder / target_name
                if target_dir.exists() and target_dir.is_dir():
                    target_dirs.append(target_dir)

    print("=" * 80)
    print(f"RAW_ROOT: {raw_root}")
    print(f"找到检查文件夹数量: {len(exam_folders)}")
    print(f"找到待删除文件夹数量: {len(target_dirs)}")
    print(f"当前模式: {'预览模式，不会删除' if dry_run else '删除模式，会真正删除'}")
    print("=" * 80)

    if len(target_dirs) == 0:
        print("没有找到需要删除的文件夹。")
        return

    for idx, target_dir in enumerate(target_dirs, start=1):
        print(f"[{idx}/{len(target_dirs)}] {target_dir}")

        if not dry_run:
            shutil.rmtree(
                target_dir,
                onerror=handle_remove_readonly
            )

    print("=" * 80)
    if dry_run:
        print("预览完成。确认路径无误后，将 DRY_RUN 改为 False 再运行。")
    else:
        print("删除完成。")


delete_target_dirs(RAW_ROOT, dry_run=DRY_RUN)